# Spring Constant Image Generator (v6.6)

Companion notebook for the Trapping Potential Mapping (TPM) pipeline.

Reads PALMTracer single-particle tracking output, computes a per-pixel spring
constant k = 1/Rg² at one or more spatial sampling scales, 
and writes per-pixel tables, descriptive statistics, and optional
TIFF maps. Two operating modes:

- `N_PROTEINS = 1` -- single-label. One discrete-colour map per zoom level.
- `N_PROTEINS = 2` -- dual-label. Three-panel figure (protein 1 in red,
  protein 2 in blue, difference overlay) per zoom level.

All user-facing settings live in the CONFIG cell below. No other cell needs
to be edited for routine use.

See `README_TPM_Analysis.docx` for the full method description,
input format, output schema, troubleshooting and version history.

## 1. Imports

Run once per kernel session.


In [26]:
import os
import re

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")           # non-interactive backend; required for headless figure saving
import matplotlib.pyplot as plt
import matplotlib.ticker
from matplotlib.colors import ListedColormap, BoundaryNorm

from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm


## 2. Configuration

Edit this cell before running. Every parameter described in README §5.

For first-time use:

1. Set `N_PROTEINS` (1 or 2).
2. Fill in `INPUT_FOLDERS` (single-label) or `DUAL_INPUT_FOLDERS` (dual-label).
3. Set `OUTPUT_ROOT`.
4. Leave `VMIN`/`VMAX` as `None` to auto-derive on a first single-label run,
   then paste the printed values back here to lock the colour scale across
   subsequent runs.

Dual-label mode requires `VMIN` and `VMAX` to be set explicitly; auto-derive
is not supported there.


In [27]:
# =============================================================================
# USER CONFIG
# =============================================================================

# Mode: 1 = single-label, 2 = dual-label
N_PROTEINS = 1

# ----- Single-label input -----------------------------------------------------
# Keys   : folder containing trcPALMTracer*.txt
# Values : path to MetaMorph .rgn boundary file, or None for no boundary
INPUT_FOLDERS = {
    # r"D:\path\to\Protein\cell1\loc and trc files": r"D:\path\to\cell1\dendrite.rgn",
    # r"D:\path\to\Protein\cell2\loc and trc files": None,
    r"D:\Publications\Spring-Constant\Input_files_SC\Homer\cell4\loc_and_trc_files":None
}

# ----- Dual-label input (used only when N_PROTEINS == 2) ----------------------
# Each value must be a 3-tuple (protein1_folder, protein2_folder, rgn_path_or_None).
# Use None for the third entry when no boundary file is available.
DUAL_INPUT_FOLDERS = {
    # "cell1": (
    #     r"D:\Publications\Spring-Constant\Input_files_SC\Dual_color\cell1\PSD95\loc_and_trc_files",
    #     r"D:\Publications\Spring-Constant\Input_files_SC\Dual_color\cell1\GluA2\loc_and_trc_files",
    #     None,
    # ),
    # "cell2": (
    #     r"D:\Publications\Spring-Constant\Input_files_SC\Dual_color\cell2\PSD95\loc_and_trc_files",
    #     r"D:\Publications\Spring-Constant\Input_files_SC\Dual_color\cell2\GluA2\loc_and_trc_files",
    #     None,
    # ),
    # "cell3": (
    #     r"D:\Publications\Spring-Constant\Input_files_SC\Dual_color\cell3\PSD95\loc_and_trc_files",
    #     r"D:\Publications\Spring-Constant\Input_files_SC\Dual_color\cell3\GluA2\loc_and_trc_files",
    #     None,
    # ),
    # "cell4": (
    #     r"D:\Publications\Spring-Constant\Input_files_SC\Dual_color\cell4\PSD95\loc_and_trc_files",
    #     r"D:\Publications\Spring-Constant\Input_files_SC\Dual_color\cell4\GluA2\loc_and_trc_files",
    #     None,
    # ),
    # "cell5": (
    #     r"D:\Publications\Spring-Constant\Input_files_SC\Dual_color\cell5\PSD95\loc_and_trc_files",
    #     r"D:\Publications\Spring-Constant\Input_files_SC\Dual_color\cell5\GluA2\loc_and_trc_files",
    #     None,
    # ),
}

# Protein display names (dual-label only)
PROTEIN1_NAME = "ProteinA"       # red panel
PROTEIN2_NAME = "ProteinB"       # blue panel

# ----- Output ----------------------------------------------------------------
OUTPUT_ROOT = r"D:\Publications\Spring-Constant\Spring Constant Home - 6 april 2026\Output\compare_old_vs_new\new"

# ----- Camera calibration ----------------------------------------------------
NM_PER_PIXEL = 160.0          # nm per camera pixel

# ----- Zoom (spatial binning) -------------------------------------------------
# bin size = NM_PER_PIXEL / zoom. e.g., zoom=4 -> 40 nm bins at 160 nm/px camera.
ZOOM_VALUES = [1, 2, 4, 8]

# ----- Track length filter ---------------------------------------------------
# Minimum localisations per track (summed across all frames).
# Suggested: 6 = ensemble SPT default; 1-2 for sparse / single-trajectory data.
MIN_POINTS = 6

# ----- Colormap scale --------------------------------------------------------
# log(k) range used for colour mapping. Leave as None to auto-derive on the
# first single-label run, then paste the printed values back to lock the scale.
# Dual-label mode REQUIRES explicit values.
VMIN = None
VMAX =  None

# ----- Single-localisation handling ------------------------------------------
# True  : segments with Rg^2 = 0 -> k = 1/(10 nm)^2  (bead / immobile control)
# False : segments with Rg^2 = 0 -> NaN (rendered white, excluded from stats)
USE_LOC_PRECISION_FLOOR = False

# ----- Single-label discrete colormap (5 bins, low k -> high k) --------------
BASE_COLORS = ["#FFE600", "#FF7518", "#8B00FF", "#0066FF", "#00CC00"]

# ----- Dual-label gradients (6 bins each, white -> dark) ---------------------
BASE_COLORS_RED  = ["#FFFFFF", "#FFCCCC", "#FF8080", "#FF3333", "#CC0000", "#660000"]
BASE_COLORS_BLUE = ["#FFFFFF", "#CCE5FF", "#80BFFF", "#3399FF", "#0055AA", "#001F4D"]

# ----- Dual-label difference thresholds --------------------------------------
# d = log(k1) - log(k2):
#   |d| <= THRESH_WEAK                          -> black  (co-confined)
#   THRESH_WEAK  <  |d| <= THRESH_STRONG        -> light red / light blue
#   |d| > THRESH_STRONG                          -> dark red / dark blue
THRESH_WEAK   = 0.3
THRESH_STRONG = 1.0

# ----- Boundary handling -----------------------------------------------------
MASK_TO_BOUNDARY   = True     # True: pixels outside .rgn polygon -> white
BOUNDARY_LINEWIDTH = 0.3      # overlay linewidth

# ----- Parallelism -----------------------------------------------------------
# None = all CPU cores. Set to 1 if you hit memory pressure.
MAX_WORKERS = 1

# ----- Image generation ------------------------------------------------------
SAVE_IMAGES = True            # False -> save .txt tables only (no TIFFs)
DPI         = 300             # 600/1200 for publication-quality TIFFs


## 3. Colormaps

Built from the hex lists in CONFIG. `ListedColormap` produces hard colour bin
boundaries; combined with `BoundaryNorm` at plot time it gives discrete,
non-interpolated colour bins. NaN pixels are mapped to white.


In [28]:
# Single-label discrete colormap
CMAP = ListedColormap(BASE_COLORS)
CMAP.set_bad(color="white")


def get_dual_cmaps():
    """Return (red, blue) ListedColormaps for dual-label rendering."""
    cmap_red  = ListedColormap(BASE_COLORS_RED)
    cmap_blue = ListedColormap(BASE_COLORS_BLUE)
    cmap_red.set_bad(color="white")
    cmap_blue.set_bad(color="white")
    return cmap_red, cmap_blue


## 4. File I/O Utilities

Trajectory loading, boundary parsing, and output writers.


In [29]:
def load_trc_files(folder):
    """
    Load every trcPALMTracer*.txt file in `folder` and concatenate them.

    Each PALMTracer file has:
      row 0  : metadata (Width, Height, nb_Planes, nb_Tracks, ...)
      row 1  : column names (Track, Plane, CentroidX(px), CentroidY(px), ...)
      row 2+ : one localisation per line

    Track IDs in different files start from 1 independently. To prevent
    different tracks being merged during per-pixel grouping, this function
    adds a cumulative offset to Track when concatenating, producing a globally
    unique `track_uid` column.

    Returns
    -------
    merged   : DataFrame of all localisations with a `track_uid` column.
    frame_w  : camera frame width  in pixels (from the metadata of file[0]).
    frame_h  : camera frame height in pixels (from the metadata of file[0]).
    """
    files = sorted(
        f for f in os.listdir(folder)
        if f.startswith("trcPALMTracer") and f.endswith(".txt")
    )
    if not files:
        raise FileNotFoundError(f"No trcPALMTracer*.txt files in {folder}")

    # Read camera frame size from the first file's metadata row.
    header  = pd.read_csv(os.path.join(folder, files[0]), sep="\t", nrows=1)
    frame_w = int(header["Width"].iloc[0])
    frame_h = int(header["Height"].iloc[0])

    dfs, offset = [], 0
    for fname in files:
        df = pd.read_csv(os.path.join(folder, fname), sep="\t", skiprows=2)
        df["track_uid"] = df["Track"] + offset
        offset += int(df["Track"].max()) + 1
        dfs.append(df)

    merged = pd.concat(dfs, ignore_index=True)
    return merged, frame_w, frame_h


def parse_protein_cell(folder):
    """
    Recover protein name and cell number from a folder path of the form
    `.../ProteinName/cellN/loc and trc files`. Returns ('ProteinName', 'N'),
    or ('ProteinName', 'NA') if the cell directory does not match `cellN`.
    """
    parts   = os.path.normpath(folder).split(os.sep)
    protein = parts[-3]
    m       = re.match(r"cell(\d+)", parts[-2])
    cell    = m.group(1) if m else "NA"
    return protein, cell


def parse_rgn(rgn_path):
    """
    Parse a MetaMorph .rgn polygon ROI.

    Field 6 stores the polygon as: 6  N  row1 col1  row2 col2  ...
    where row corresponds to the camera y axis and col to the camera x axis.
    This function swaps them so the returned coordinates are (x, y) in
    standard image convention.

    Returns
    -------
    bnd_x_px, bnd_y_px : 1-D float arrays of polygon vertices in camera pixels.
    """
    raw    = open(rgn_path).read()
    fields = re.split(r",\s*", raw.strip())
    field6 = [f for f in fields if f.strip().startswith("6 ")][0]
    nums   = list(map(int, field6.strip().split()))
    coords = nums[2:]                                  # skip field id and point count
    rgn_rows = np.array(coords[0::2], dtype=float)
    rgn_cols = np.array(coords[1::2], dtype=float)
    bnd_x_px = rgn_rows                                # row -> x
    bnd_y_px = rgn_cols                                # col -> y
    return bnd_x_px, bnd_y_px


def parse_spine_rois(rgn_path, nm_per_pixel=160.0):
    """
    Parse a MetaMorph .rgn file containing one or more rectangle ROIs.

    Each line is one ROI with:
      field 2 : centre (row, col) in camera pixels
      field 6 : (count, full_height, full_width) in pixels
      field 7 : (count, roi_id)

    Returns a list of dicts with keys: id, x_min, x_max, y_min, y_max (all in um).
    Currently unused by the main pipeline; kept for ad-hoc spine analysis.
    """
    lines = open(rgn_path).read().strip().splitlines()
    rois  = []
    for line in lines:
        line = line.strip()
        if not line:
            continue
        fields = re.split(r",\s*", line)
        try:
            f2 = [f for f in fields if f.strip().startswith("2 ")][0].strip().split()
            f6 = [f for f in fields if f.strip().startswith("6 ")][0].strip().split()
            f7 = [f for f in fields if f.strip().startswith("7 ")][0].strip().split()
        except IndexError:
            continue
        if len(f2) < 3 or len(f6) < 4:
            continue
        centre_row = int(f2[1]); centre_col = int(f2[2])
        half_h     = int(f6[2]) / 2;  half_w  = int(f6[3]) / 2
        roi_id     = int(f7[1])
        rois.append({
            "id":    roi_id,
            "x_min": (centre_row - half_h) * nm_per_pixel / 1000,
            "x_max": (centre_row + half_h) * nm_per_pixel / 1000,
            "y_min": (centre_col - half_w) * nm_per_pixel / 1000,
            "y_max": (centre_col + half_w) * nm_per_pixel / 1000,
        })
    return rois


def save_spring_constants(df, save_path):
    """Write the per-pixel spring constant DataFrame as a tab-delimited .txt."""
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    header = "<xid>\t<yid>\t<x in nm>\t<y in nm>\t<Rg>\t<Spring Const k>\t<log10(k)>"
    with open(save_path, "w") as f:
        f.write(header + "\n")
    df.to_csv(save_path, sep="\t", index=False, header=False, mode="a")


def compute_logK_stats(df_spring, zoom, cell_label):
    """
    Descriptive statistics of log(k) for one (cell, zoom) combination.

    Returns a dict with keys: Cell, Zoom, N_pixels, Min, P2, P5, P25, Mean,
    SEM, Median, P75, P95, P98, Max. SEM is computed with ddof=1.
    """
    vals = df_spring["log10k"].replace([np.inf, -np.inf], np.nan).dropna()
    n = len(vals)
    if n == 0:
        return dict(Cell=cell_label, Zoom=zoom, N_pixels=0,
                    Min=np.nan, P2=np.nan, P5=np.nan, P25=np.nan,
                    Mean=np.nan, SEM=np.nan,
                    Median=np.nan, P75=np.nan, P95=np.nan, P98=np.nan, Max=np.nan)
    sem = float(vals.std(ddof=1) / np.sqrt(n)) if n > 1 else np.nan
    return dict(
        Cell=cell_label, Zoom=zoom, N_pixels=n,
        Min=float(vals.min()),
        P2=float(np.percentile(vals,  2)),
        P5=float(np.percentile(vals,  5)),
        P25=float(np.percentile(vals, 25)),
        Mean=float(vals.mean()),
        SEM=sem,
        Median=float(vals.median()),
        P75=float(np.percentile(vals, 75)),
        P95=float(np.percentile(vals, 95)),
        P98=float(np.percentile(vals, 98)),
        Max=float(vals.max()),
    )


def write_stats_excel(records, save_path):
    """
    Write the per-(cell, zoom) descriptive statistics to a formatted Excel file.

    `records` is a list of dicts produced by compute_logk_stats.
    `save_path` must end in .xlsx.
    """
    import openpyxl
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
    from openpyxl.utils import get_column_letter

    col_order = ["Cell", "Zoom", "N_pixels",
                 "Min", "P2", "P5", "P25", "Mean", "SEM",
                 "Median", "P75", "P95", "P98", "Max"]
    df = pd.DataFrame(records, columns=col_order)
    df = df.sort_values(["Cell", "Zoom"]).reset_index(drop=True)

    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = "logk Stats Summary"

    # Header row
    hdr_fill = PatternFill("solid", fgColor="2F4F8F")
    hdr_font = Font(bold=True, color="FFFFFF")
    for col_i, h in enumerate(col_order, 1):
        c = ws.cell(row=1, column=col_i, value=h)
        c.fill = hdr_fill
        c.font = hdr_font
        c.alignment = Alignment(horizontal="center")

    # Data rows
    thin   = Side(style="thin", color="CCCCCC")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    for row_i, row in df.iterrows():
        for col_i, col_name in enumerate(col_order, 1):
            cell = ws.cell(row=row_i + 2, column=col_i, value=row[col_name])
            cell.border = border
            if col_name in ("Zoom", "N_pixels"):
                cell.alignment = Alignment(horizontal="center")
            elif col_name != "Cell":
                cell.number_format = "0.000"
                cell.alignment = Alignment(horizontal="center")

    # Column widths
    for col_i, col_name in enumerate(col_order, 1):
        col_vals = [str(v) for v in df[col_name]]
        max_len  = max(len(col_name), max((len(v) for v in col_vals), default=0))
        ws.column_dimensions[get_column_letter(col_i)].width = max(10, min(max_len + 2, 30))

    save_dir = os.path.dirname(save_path)
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
    wb.save(save_path)
    print(f"  -> Stats Excel saved: {save_path}")


## 5. Spring Constant Computation

Method: Each track is split by pixel bin, Rg² is computed
within each segment, and segments sharing a pixel are averaged.

The implementation uses pandas `transform()` and `agg()` so the inner loops
stay in compiled code; the previous `groupby.apply()` Python-lambda variant
(v6.0) is slow on large datasets.

`compute_spring_constants` returns a single DataFrame. Do not unpack it as
`df, _ = compute_spring_constants(...)`.


In [30]:
def compute_spring_constants(df, zoom=1.0, nm_per_pixel=160.0,
                              min_points=6, use_floor=False):
    """
    Compute per-pixel spring constants k = 1/Rg^2 from a localisation table.

    For each pixel bin and each track segment that falls inside that bin,
    Rg^2 is the mean squared displacement of localisations from the segment
    centroid. Segments sharing a pixel are averaged. k = 1/Rg^2 (nm^-2).

    Parameters
    ----------
    df           : DataFrame with `CentroidX(px)`, `CentroidY(px)`, `track_uid`.
    zoom         : spatial binning factor. zoom=2 with nm_per_pixel=160 gives
                   80 nm bins.
    nm_per_pixel : camera calibration.
    min_points   : minimum localisations per track (summed across all frames).
                   Tracks shorter than this are removed before pixel binning.
    use_floor    : if True, single-localisation segments (Rg^2 = 0) are
                   floored to k = 1/(10 nm)^2 (bead/immobile data).
                   if False (default for SPT), they become NaN -> white pixel.

    Returns
    -------
    DataFrame with columns: xid, yid, x_nm, y_nm, Rg, k, log10k.
    """
    unit  = nm_per_pixel / zoom
    unit2 = unit ** 2

    x      = df["CentroidX(px)"].to_numpy(dtype=float)
    y      = df["CentroidY(px)"].to_numpy(dtype=float)
    tracks = df["track_uid"].to_numpy()

    # Assign every localisation to a pixel bin in the zoom-scaled grid.
    px = np.floor(x * zoom).astype(np.int32)
    py = np.floor(y * zoom).astype(np.int32)

    tmp = pd.DataFrame({"x": x, "y": y, "track": tracks, "px": px, "py": py})

    # Drop tracks shorter than `min_points` (total length over all frames).
    track_sizes = tmp.groupby("track").size()
    tmp = tmp[tmp["track"].isin(track_sizes[track_sizes >= min_points].index)].copy()
    if tmp.empty:
        return pd.DataFrame(columns=["xid", "yid", "x_nm", "y_nm", "Rg", "k", "log10k"])

    # Per-localisation squared deviation from the (track, pixel) centroid.
    # `transform("mean")` broadcasts the group mean back to row level without
    # an explicit Python loop.
    grp_keys = ["track", "px", "py"]
    tmp["x_mean"] = tmp.groupby(grp_keys)["x"].transform("mean")
    tmp["y_mean"] = tmp.groupby(grp_keys)["y"].transform("mean")
    tmp["d2"]     = (tmp["x"] - tmp["x_mean"]) ** 2 + (tmp["y"] - tmp["y_mean"]) ** 2

    # One Rg^2 per (track, pixel) segment.
    tp_agg = tmp.groupby(grp_keys).agg(rg2=("d2", "mean")).reset_index()
    if tp_agg.empty:
        return pd.DataFrame(columns=["xid", "yid", "x_nm", "y_nm", "Rg", "k", "log10k"])

    # Average segment Rg^2 across all tracks visiting the same pixel.
    pixel_agg = tp_agg.groupby(["px", "py"]).agg(mean_rg2=("rg2", "mean")).reset_index()

    # Convert from pixel^2 units to physical Rg^2 in nm^2.
    rg_nm2 = pixel_agg["mean_rg2"].to_numpy() * unit2

    # Handle single-localisation segments (Rg^2 = 0 exactly).
    if use_floor:
        rg_nm2 = np.where(rg_nm2 == 0, 100.0, rg_nm2)        # (10 nm)^2 floor
    else:
        rg_nm2 = np.where(rg_nm2 == 0, np.nan, rg_nm2)

    k    = np.where(np.isfinite(rg_nm2), 1.0 / rg_nm2, np.nan)
    logk = np.where(k > 0, np.log10(k), np.nan)

    return pd.DataFrame({
        "xid":    pixel_agg["px"].to_numpy(),
        "yid":    pixel_agg["py"].to_numpy(),
        "x_nm":   pixel_agg["px"].to_numpy() * unit,
        "y_nm":   pixel_agg["py"].to_numpy() * unit,
        "Rg":     rg_nm2,
        "k":      k,
        "log10k": logk,
    })


def build_grid(df, zoom, nm_per_pixel=160.0, frame_w=256, frame_h=256):
    """
    Project the per-pixel k values onto a 2-D grid the size of the full
    camera frame at the given zoom factor.

    Using the full frame (rather than the data bounding box) keeps the image
    aspect ratio consistent across cells regardless of where the tracks lie.

    Returns
    -------
    xx : 1-D array of x-axis positions in um (length grid_w).
    yy : 1-D array of y-axis positions in um (length grid_h).
    zz : 2-D k array of shape (grid_w, grid_h), NaN where no data.
    """
    unit   = nm_per_pixel / zoom
    grid_w = int(np.floor(frame_w * zoom))
    grid_h = int(np.floor(frame_h * zoom))

    xids = df["xid"].to_numpy(dtype=int)
    yids = df["yid"].to_numpy(dtype=int)
    rg   = df["Rg"].to_numpy(dtype=float)

    zz = np.zeros((grid_w, grid_h))
    valid = (xids < grid_w) & (yids < grid_h)
    zz[xids[valid], yids[valid]] = np.where(rg[valid] > 0, 1.0 / rg[valid], np.nan)

    xx = np.arange(grid_w) * unit / 1000   # um
    yy = np.arange(grid_h) * unit / 1000

    return xx, yy, zz


## 6. Boundary Masking

Used by the plot functions when `MASK_TO_BOUNDARY = True`. Uses
`skimage.draw.polygon` for fast rasterisation and falls back to
`matplotlib.path.Path` when scikit-image isn't installed.


In [31]:
def apply_boundary_mask(norm_arr, xx, yy, bnd_x_um, bnd_y_um):
    """
    Set pixels outside the dendrite polygon to NaN (renders as white).

    Parameters
    ----------
    norm_arr : (H, W) float array, NaN where there is no data.
    xx       : 1-D x positions in um (columns).
    yy       : 1-D y positions in um (rows).
    bnd_x_um : polygon vertex x coordinates in um.
    bnd_y_um : polygon vertex y coordinates in um.

    Returns
    -------
    norm_arr : the same array with outside pixels set to NaN (modified in place).
    """
    H, W = norm_arr.shape
    try:
        from skimage.draw import polygon as sk_polygon
        # Map um boundary coordinates to (row, col) indices in norm_arr.
        row_c = np.interp(bnd_y_um, yy, np.arange(len(yy)))
        col_c = np.interp(bnd_x_um, xx, np.arange(len(xx)))
        rr, cc = sk_polygon(row_c, col_c, shape=(H, W))
        outside = np.ones((H, W), dtype=bool)
        outside[rr, cc] = False
    except ImportError:
        from matplotlib.path import Path
        poly = Path(list(zip(bnd_x_um, bnd_y_um)))
        X_g, Y_g = np.meshgrid(xx, yy)
        coords  = np.column_stack([X_g.ravel(), Y_g.ravel()])
        outside = ~poly.contains_points(coords).reshape(norm_arr.shape)
    norm_arr[outside] = np.nan
    return norm_arr


## 7. Single-Label Plotting

`plot_spring_constant_map()` writes one TIFF per zoom level using the
discrete `ListedColormap` + `BoundaryNorm` combination. Raw log10(k) is
upsampled 4x with bilinear interpolation before colour mapping, which keeps
the colour bins discrete while giving a smooth spatial appearance.


In [32]:
def plot_spring_constant_map(
    xx, yy, zz,
    vmin, vmax,
    save_path,
    title="",
    bnd_x_um=None, bnd_y_um=None,
    mask_to_boundary=False,
):
    """
    Save a single-label spring constant map as a TIFF.

    Rendering pipeline:
      1. log10(k) transform; NaN where k <= 0 or no data.
      2. Optional boundary mask (NaN outside the .rgn polygon).
      3. 4x bilinear upsampling of the raw log10(k) data (NaN propagated via
         a nearest-neighbour mask).
      4. Discrete colour mapping with ListedColormap + BoundaryNorm.
      5. Boundary polygon overlaid in physical (µm) coordinates.
    """
    from scipy.ndimage import zoom as ndimage_zoom

    # log-transform; preserve NaN where k is not positive.
    zz_log   = np.log10(np.where(zz > 0, zz, np.nan))
    zz_log_T = np.where(np.isfinite(zz_log), zz_log, np.nan).T   # (H, W)

    display_arr = np.where(np.isfinite(zz_log_T), zz_log_T, np.nan)

    if mask_to_boundary and bnd_x_um is not None:
        display_arr = apply_boundary_mask(
            display_arr, xx, yy,
            np.array(bnd_x_um), np.array(bnd_y_um),
        )

    # Upsample raw data first, then colour-map, so colour bins stay discrete.
    SCALE_UP = 4
    nan_mask  = ~np.isfinite(display_arr)
    filled    = np.where(nan_mask, vmin, display_arr)
    upsampled = ndimage_zoom(filled, zoom=SCALE_UP, order=1)                   # bilinear
    nan_up    = ndimage_zoom(nan_mask.astype(float), zoom=SCALE_UP, order=0) > 0.5
    upsampled = np.where(nan_up, np.nan, upsampled)
    display   = np.ma.masked_invalid(upsampled)

    # Discrete colour scale.
    cmap_plot = CMAP
    cmap_plot.set_bad(color="white")
    n_bins         = len(BASE_COLORS)
    bin_edges_norm = np.linspace(vmin, vmax, n_bins + 1)
    norm_discrete  = BoundaryNorm(bin_edges_norm, ncolors=n_bins)

    # Figure sizing scales with data aspect ratio.
    x_range = xx[-1] - xx[0]
    y_range = yy[-1] - yy[0]
    scale   = 6.0 / max(x_range, y_range)
    fig, ax = plt.subplots(figsize=(x_range * scale + 1.5, y_range * scale))
    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")

    ax.imshow(display, origin="upper",
              extent=[xx[0], xx[-1], yy[-1], yy[0]],
              cmap=cmap_plot, norm=norm_discrete,
              interpolation="nearest", aspect="equal")

    if bnd_x_um is not None:
        ax.plot(np.append(bnd_x_um, bnd_x_um[0]),
                np.append(bnd_y_um, bnd_y_um[0]),
                color="black", linewidth=BOUNDARY_LINEWIDTH, alpha=0.85)

    ax.set_xlabel("X (µm)", fontsize=10)
    ax.set_ylabel("Y (µm)", fontsize=10)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.tick_params(labelsize=8)
    for spine in ax.spines.values():
        spine.set_edgecolor("#cccccc")

    # Colorbar with ticks at the bin edges.
    sm = plt.cm.ScalarMappable(
        cmap=CMAP, norm=BoundaryNorm(bin_edges_norm, ncolors=n_bins)
    )
    sm.set_array([])
    cb = fig.colorbar(sm, ax=ax, shrink=0.92, pad=0.01,
                      boundaries=bin_edges_norm, ticks=bin_edges_norm)
    cb.set_label("log₁₀(k)", fontsize=10)
    cb.ax.tick_params(labelsize=8)
    cb.ax.yaxis.set_major_formatter(matplotlib.ticker.FormatStrFormatter("%.1f"))

    os.makedirs(os.path.dirname(os.path.abspath(save_path)), exist_ok=True)
    fig.savefig(save_path, dpi=DPI, bbox_inches="tight", facecolor="white",
                format="tiff", pil_kwargs={"compression": "tiff_adobe_deflate"})
    plt.close(fig)


## 8. Dual-Label Plotting

Three-panel figure: protein 1 (red gradient), protein 2 (blue gradient), and
a difference overlay `log10(k1) - log10(k2)` rendered with a five-colour
scheme using `THRESH_WEAK` and `THRESH_STRONG` from CONFIG. Pixels where
only one protein has data fall back to that protein's gradient colour.


In [33]:
def plot_dual_label_map(
    xx, yy, zz1, zz2,
    vmin, vmax,
    save_path,
    tag="", zoom=1,
    bnd_x_um=None, bnd_y_um=None,
    mask_to_boundary=False,
    label1="Protein1", label2="Protein2",
    thresh_weak=0.3, thresh_strong=1.0,
):
    """
    Save a three-panel TIFF: red-mapped P1, blue-mapped P2, and a 5-colour
    difference panel based on log10(k1) - log10(k2).

    Parameters
    ----------
    xx, yy           : 1-D coordinate arrays in µm (same grid for both proteins).
    zz1, zz2         : 2-D k grids for protein 1 and 2 (from build_grid).
    vmin, vmax       : shared log10(k) range for panels 1 and 2.
    save_path        : output .tiff path.
    tag, zoom        : title metadata.
    bnd_x_um, bnd_y_um : polygon vertices in µm (optional).
    mask_to_boundary : True -> set pixels outside the polygon to NaN.
    label1, label2   : protein names used in titles, colorbars, legend.
    thresh_weak,
    thresh_strong    : difference-map thresholds (see CONFIG).
    """
    cmap_red, cmap_blue = get_dual_cmaps()

    # RGBA palette for the 5-colour difference panel.
    DARK_RED   = np.array([180,   0,   0, 255], dtype=np.uint8)
    LIGHT_RED  = np.array([255, 160, 160, 255], dtype=np.uint8)
    BLACK      = np.array([ 20,  20,  20, 255], dtype=np.uint8)
    LIGHT_BLUE = np.array([160, 180, 255, 255], dtype=np.uint8)
    DARK_BLUE  = np.array([  0,   0, 180, 255], dtype=np.uint8)

    def log_norm(zz):
        """log10(k), then linearly normalise to [0, 1] using vmin/vmax."""
        zz_log   = np.log10(np.where(zz > 0, zz, np.nan))
        zz_log_T = np.where(np.isfinite(zz_log), zz_log, np.nan).T
        norm = np.clip(
            (np.nan_to_num(zz_log_T, nan=vmin) - vmin) / (vmax - vmin), 0, 1
        ).astype(float)
        norm[~np.isfinite(zz_log_T)] = np.nan
        return norm, zz_log_T

    norm1, logk1 = log_norm(zz1)
    norm2, logk2 = log_norm(zz2)

    if mask_to_boundary and bnd_x_um is not None:
        norm1 = apply_boundary_mask(norm1, xx, yy, bnd_x_um, bnd_y_um)
        norm2 = apply_boundary_mask(norm2, xx, yy, bnd_x_um, bnd_y_um)
        logk1[~np.isfinite(norm1)] = np.nan
        logk2[~np.isfinite(norm2)] = np.nan

    # Build the difference panel as an explicit RGBA array.
    H, W      = norm1.shape
    diff_rgba = np.full((H, W, 4), 255, dtype=np.uint8)            # default white
    has1 = np.isfinite(norm1)
    has2 = np.isfinite(norm2)
    diff = logk1 - logk2                                           # NaN where either is missing

    both = has1 & has2 & np.isfinite(diff)
    diff_rgba[both & (diff >  thresh_strong)]                            = DARK_RED
    diff_rgba[both & (diff >  thresh_weak)  & (diff <= thresh_strong)]   = LIGHT_RED
    diff_rgba[both & (diff >= -thresh_weak) & (diff <= thresh_weak)]     = BLACK
    diff_rgba[both & (diff <  -thresh_weak) & (diff >= -thresh_strong)]  = LIGHT_BLUE
    diff_rgba[both & (diff <  -thresh_strong)]                           = DARK_BLUE

    # Where only one protein has data, fall back to that protein's gradient.
    only1 = has1 & ~has2
    if only1.any():
        diff_rgba[only1] = (cmap_red(norm1[only1])  * 255).astype(np.uint8)
    only2 = ~has1 & has2
    if only2.any():
        diff_rgba[only2] = (cmap_blue(norm2[only2]) * 255).astype(np.uint8)

    # Figure layout.
    x_range = xx[-1] - xx[0]
    y_range = yy[-1] - yy[0]
    scale   = 5.0 / max(x_range, y_range)
    pw, ph  = x_range * scale + 1.4, y_range * scale

    fig, axes = plt.subplots(1, 3, figsize=(pw * 3, ph),
                             gridspec_kw={"wspace": 0.12})
    fig.patch.set_facecolor("white")
    extent = [xx[0], xx[-1], yy[-1], yy[0]]

    axes[0].set_facecolor("white")
    axes[0].imshow(norm1, origin="upper", extent=extent,
                   cmap=cmap_red, vmin=0, vmax=1,
                   interpolation="bilinear", aspect="equal")

    axes[1].set_facecolor("white")
    axes[1].imshow(norm2, origin="upper", extent=extent,
                   cmap=cmap_blue, vmin=0, vmax=1,
                   interpolation="bilinear", aspect="equal")

    axes[2].set_facecolor("white")
    axes[2].imshow(diff_rgba, origin="upper", extent=extent,
                   interpolation="bilinear", aspect="equal")

    titles = [label1, label2, f"{label1} vs {label2}"]
    for ax, title in zip(axes, titles):
        if bnd_x_um is not None:
            ax.plot(np.append(bnd_x_um, bnd_x_um[0]),
                    np.append(bnd_y_um, bnd_y_um[0]),
                    color="black", linewidth=0.4, alpha=0.85)
        ax.set_title(title, fontsize=10, fontweight="bold", pad=3)
        ax.set_xlabel("X (µm)", fontsize=8)
        ax.tick_params(labelsize=7)
        for spine in ax.spines.values():
            spine.set_edgecolor("#cccccc")
    axes[0].set_ylabel("Y (µm)", fontsize=8)
    axes[1].set_yticklabels([])
    axes[2].set_yticklabels([])

    # Gradient colorbars for panels 1 and 2.
    bin_edges = np.linspace(vmin, vmax, 6)
    for cmap_cb, ax, lbl in [
        (cmap_red,  axes[0], f"log₁₀(k)  [{label1}]"),
        (cmap_blue, axes[1], f"log₁₀(k)  [{label2}]"),
    ]:
        sm = plt.cm.ScalarMappable(cmap=cmap_cb, norm=plt.Normalize(vmin=vmin, vmax=vmax))
        sm.set_array([])
        cb = fig.colorbar(sm, ax=ax, shrink=0.85, pad=0.02,
                          boundaries=bin_edges, ticks=bin_edges)
        cb.set_label(lbl, fontsize=8)
        cb.ax.tick_params(labelsize=7)

    # Legend for the difference panel.
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor=tuple(DARK_RED[:3]   / 255),
              label=f"{label1} >> {label2}  (diff > +{thresh_strong})"),
        Patch(facecolor=tuple(LIGHT_RED[:3]  / 255),
              label=f"{label1} > {label2}  (diff +{thresh_weak} to +{thresh_strong})"),
        Patch(facecolor=tuple(BLACK[:3]      / 255),
              label=f"Co-confined  (|diff| ≤ {thresh_weak})"),
        Patch(facecolor=tuple(LIGHT_BLUE[:3] / 255),
              label=f"{label2} > {label1}  (diff -{thresh_strong} to -{thresh_weak})"),
        Patch(facecolor=tuple(DARK_BLUE[:3]  / 255),
              label=f"{label2} >> {label1}  (diff < -{thresh_strong})"),
    ]
    axes[2].legend(handles=legend_elements, loc="lower right",
                   fontsize=6, framealpha=0.85,
                   title="log₁₀(k₁)−log₁₀(k₂)",
                   title_fontsize=6)

    fig.suptitle(f"{tag}  {zoom}×", fontsize=11, fontweight="bold", y=1.01)
    os.makedirs(os.path.dirname(os.path.abspath(save_path)), exist_ok=True)
    fig.savefig(save_path, dpi=DPI, bbox_inches="tight", facecolor="white",
                format="tiff", pil_kwargs={"compression": "tiff_adobe_deflate"})
    plt.close(fig)


## 9. Single-Label Pipeline

`run_pipeline()` processes one cell folder: load, compute k at every zoom,
save the per-pixel table, collect statistics, and optionally save the TIFF
map. `_worker()` is the top-level entry point used by `ProcessPoolExecutor`
in `main()`.

When `save_images=False`, computation still runs at all zoom levels but no
TIFFs are written.


In [34]:
def run_pipeline(
    input_folder, output_folder,
    zoom_values, nm_per_pixel,
    vmin, vmax, min_points,
    bnd_x_um=None, bnd_y_um=None,
    mask_to_boundary=False,
    save_images=True,
):
    """
    Single-label pipeline for one cell folder.

    Returns a list of stats dicts (one per zoom level) for later aggregation
    by `main()`.
    """
    protein, cell = parse_protein_cell(input_folder)
    tag = f"{protein}_cell{cell}"

    df_all, frame_w, frame_h = load_trc_files(input_folder)
    print(f"  {tag}: {len(df_all):,} localisations, "
          f"{df_all['track_uid'].nunique():,} unique tracks  "
          f"[frame {frame_w}x{frame_h} px]")

    stats_records = []
    for zoom in zoom_values:
        df_spring = compute_spring_constants(
            df_all, zoom=zoom,
            nm_per_pixel=nm_per_pixel,
            min_points=min_points,
            use_floor=USE_LOC_PRECISION_FLOOR,
        )

        # Per-pixel table.
        txt_path = os.path.join(output_folder,
                                f"{tag}_zoom{zoom}x_spring_constants.txt")
        save_spring_constants(df_spring, txt_path)

        stats_records.append(compute_logK_stats(df_spring, zoom, tag))

        xx, yy, zz = build_grid(df_spring, zoom, nm_per_pixel,
                                frame_w=frame_w, frame_h=frame_h)
        valid = df_spring["log10k"].replace([np.inf, -np.inf], np.nan).dropna()
        med   = float(valid.median()) if len(valid) else np.nan

        print(f"    zoom={zoom}x ({nm_per_pixel/zoom:.0f}nm/px): "
              f"{len(df_spring)} pixels  log10k median={med:.2f}")

        if save_images:
            tiff_path = os.path.join(output_folder,
                                     f"{tag}_zoom{zoom}x_map.tiff")
            plot_spring_constant_map(
                xx, yy, zz,
                vmin=vmin, vmax=vmax,
                save_path=tiff_path,
                title=f"{tag}  {zoom}×",
                bnd_x_um=bnd_x_um, bnd_y_um=bnd_y_um,
                mask_to_boundary=mask_to_boundary,
            )
            print(f"      -> saved {tiff_path}")

    return stats_records


def save_config_snapshot(output_root):
    """
    Write a plain-text record of the CONFIG values used in this run to
    `output_root/config_snapshot.txt`. Called once at the start of main().
    """
    config_lines = [
        "Spring Constant Generator -- CONFIG snapshot",
        "=" * 50,
        f"N_PROTEINS              = {N_PROTEINS}",
        f"NM_PER_PIXEL            = {NM_PER_PIXEL}",
        f"ZOOM_VALUES             = {ZOOM_VALUES}",
        f"MIN_POINTS              = {MIN_POINTS}",
        f"VMIN                    = {VMIN}",
        f"VMAX                    = {VMAX}",
        f"USE_LOC_PRECISION_FLOOR = {USE_LOC_PRECISION_FLOOR}",
        f"BASE_COLORS             = {BASE_COLORS}",
        f"MASK_TO_BOUNDARY        = {MASK_TO_BOUNDARY}",
        f"SAVE_IMAGES             = {SAVE_IMAGES}",
        f"DPI                     = {DPI}",
        f"MAX_WORKERS             = {MAX_WORKERS}",
        f"BOUNDARY_LINEWIDTH      = {BOUNDARY_LINEWIDTH}",
        f"THRESH_WEAK             = {THRESH_WEAK}",
        f"THRESH_STRONG           = {THRESH_STRONG}",
        f"PROTEIN1_NAME           = {PROTEIN1_NAME}",
        f"PROTEIN2_NAME           = {PROTEIN2_NAME}",
        "",
        "INPUT_FOLDERS:",
    ]
    for folder, rgn in INPUT_FOLDERS.items():
        config_lines.append(f"  {folder}")
        config_lines.append(f"    rgn = {rgn}")
    config_lines += ["", "DUAL_INPUT_FOLDERS:"]
    for cell_name, (f1, f2, rgn) in DUAL_INPUT_FOLDERS.items():
        config_lines.append(f"  {cell_name}")
        config_lines.append(f"    protein1 = {f1}")
        config_lines.append(f"    protein2 = {f2}")
        config_lines.append(f"    rgn      = {rgn}")

    os.makedirs(output_root, exist_ok=True)
    path = os.path.join(output_root, "config_snapshot.txt")
    with open(path, "w") as f:
        f.write("\n".join(config_lines))
    print(f"  -> Config snapshot saved: {path}")


def _worker(args):
    """Top-level wrapper so ProcessPoolExecutor can pickle the call."""
    (
        input_folder, output_folder,
        zoom_values, nm_per_pixel,
        vmin, vmax, min_points,
        bnd_x_um, bnd_y_um,
        mask_to_boundary,
        save_images,
    ) = args
    stats_records = run_pipeline(
        input_folder, output_folder,
        zoom_values, nm_per_pixel,
        vmin, vmax, min_points,
        bnd_x_um, bnd_y_um,
        mask_to_boundary,
        save_images,
    )
    return input_folder, stats_records


## 10. Dual-Label Pipeline

`run_dual_pipeline()` is the dual-label counterpart of `run_pipeline()`. It
loads both protein folders, computes k independently for each, and saves a
three-panel TIFF per zoom level. The two protein grids share `frame_w`,
`frame_h` from protein 1's metadata so the panels are pixel-aligned.

No statistics Excel is generated in dual-label mode.


In [35]:
def run_dual_pipeline(
    cell_name, folder1, folder2,
    output_folder,
    zoom_values, nm_per_pixel,
    vmin, vmax, min_points,
    bnd_x_um=None, bnd_y_um=None,
    mask_to_boundary=False,
    label1="Protein1", label2="Protein2",
    thresh_weak=0.3, thresh_strong=1.0,
    save_images=True,
):
    """
    Dual-label pipeline for one cell.

    Both protein folders are loaded and processed independently. The k grids
    share frame dimensions from protein 1, so all three output panels are
    pixel-aligned. No txt tables or stats Excel are produced in this mode.
    """
    os.makedirs(output_folder, exist_ok=True)

    df1, frame_w, frame_h = load_trc_files(folder1)
    df2, _,       _       = load_trc_files(folder2)
    print(f"  {cell_name}: {label1} {len(df1):,} locs | "
          f"{label2} {len(df2):,} locs  [frame {frame_w}x{frame_h}px]")

    for zoom in zoom_values:
        df_s1 = compute_spring_constants(
            df1, zoom=zoom,
            nm_per_pixel=nm_per_pixel, min_points=min_points,
            use_floor=USE_LOC_PRECISION_FLOOR,
        )
        df_s2 = compute_spring_constants(
            df2, zoom=zoom,
            nm_per_pixel=nm_per_pixel, min_points=min_points,
            use_floor=USE_LOC_PRECISION_FLOOR,
        )

        xx, yy, zz1 = build_grid(df_s1, zoom, nm_per_pixel,
                                  frame_w=frame_w, frame_h=frame_h)
        _,  _,  zz2 = build_grid(df_s2, zoom, nm_per_pixel,
                                  frame_w=frame_w, frame_h=frame_h)

        print(f"    zoom={zoom}x: {label1} {len(df_s1)} px | "
              f"{label2} {len(df_s2)} px")

        if save_images:
            save_path = os.path.join(output_folder,
                                     f"{cell_name}_zoom{zoom}x_dual.tiff")
            plot_dual_label_map(
                xx, yy, zz1, zz2,
                vmin=vmin, vmax=vmax,
                save_path=save_path,
                tag=cell_name, zoom=zoom,
                bnd_x_um=bnd_x_um, bnd_y_um=bnd_y_um,
                mask_to_boundary=mask_to_boundary,
                label1=label1, label2=label2,
                thresh_weak=thresh_weak, thresh_strong=thresh_strong,
            )
            print(f"      -> saved {save_path}")


## 11. Main Orchestrators

- `main()` runs the single-label pipeline for every entry in `INPUT_FOLDERS`,
  auto-derives `VMIN`/`VMAX` from data if either is `None`, and writes the
  combined statistics Excel.
- `main_dual()` runs the dual-label pipeline for every entry in
  `DUAL_INPUT_FOLDERS`. Requires `VMIN` and `VMAX` to be set explicitly.

Both write `config_snapshot.txt` to `OUTPUT_ROOT` at the start of the run.


In [36]:
def main(
    input_folders=None,
    output_root=OUTPUT_ROOT,
    zoom_values=ZOOM_VALUES,
    nm_per_pixel=NM_PER_PIXEL,
    vmin=VMIN, vmax=VMAX,
    min_points=MIN_POINTS,
    max_workers=MAX_WORKERS,
    mask_to_boundary=MASK_TO_BOUNDARY,
    save_images=SAVE_IMAGES,
):
    """
    Single-label entry point. Reads from CONFIG by default; edit CONFIG
    rather than passing arguments here for routine use.
    """
    if input_folders is None:
        input_folders = INPUT_FOLDERS
    os.makedirs(output_root, exist_ok=True)
    save_config_snapshot(output_root)
    all_stats = []

    # Auto-derive VMIN/VMAX from the data if either is missing.
    if vmin is None or vmax is None:
        lowest_zoom  = min(zoom_values)
        highest_zoom = max(zoom_values)
        print("\nAuto-deriving colormap scale...")
        print(f"  VMIN <- min log10(k) at zoom={lowest_zoom}x across all cells")
        print(f"  VMAX <- max log10(k) at zoom={highest_zoom}x across all cells")
        vmin_candidates, vmax_candidates = [], []
        for folder in input_folders.keys():
            df_all, _, _ = load_trc_files(folder)

            df_low = compute_spring_constants(
                df_all, zoom=lowest_zoom,
                nm_per_pixel=nm_per_pixel, min_points=min_points,
                use_floor=USE_LOC_PRECISION_FLOOR,
            )
            v_low = df_low["log10k"].replace([np.inf, -np.inf], np.nan).dropna()
            if len(v_low):
                vmin_candidates.append(float(v_low.min()))

            df_high = compute_spring_constants(
                df_all, zoom=highest_zoom,
                nm_per_pixel=nm_per_pixel, min_points=min_points,
                use_floor=USE_LOC_PRECISION_FLOOR,
            )
            v_high = df_high["log10k"].replace([np.inf, -np.inf], np.nan).dropna()
            if len(v_high):
                vmax_candidates.append(float(v_high.max()))

        vmin = float(np.min(vmin_candidates))
        vmax = float(np.max(vmax_candidates))
        print(f"  -> VMIN={vmin:.3f}, VMAX={vmax:.3f}")
        print(f"  Paste VMIN={vmin:.2f} and VMAX={vmax:.2f} into CONFIG to lock the scale.")

    print(f"\nColormap scale: log10(k) in [{vmin:.2f}, {vmax:.2f}]")
    print(f"Image generation: {'ON' if save_images else 'OFF (txt tables only)'}")

    # Build the per-folder task list.
    tasks = []
    for folder, rgn_path in input_folders.items():
        protein, cell = parse_protein_cell(folder)
        out = os.path.join(output_root, f"{protein}_cell{cell}")
        os.makedirs(out, exist_ok=True)

        bnd_x_um = bnd_y_um = None
        if rgn_path and os.path.exists(rgn_path):
            bnd_x_px, bnd_y_px = parse_rgn(rgn_path)
            bnd_x_um = (bnd_x_px * nm_per_pixel / 1000).tolist()
            bnd_y_um = (bnd_y_px * nm_per_pixel / 1000).tolist()
            print(f"  Loaded ROI for {protein} cell{cell}: {len(bnd_x_px)} pts")

        tasks.append((
            folder, out, zoom_values, nm_per_pixel,
            vmin, vmax, min_points,
            bnd_x_um, bnd_y_um,
            mask_to_boundary,
            save_images,
        ))

    # Run serial when forced, or when there is only one folder anyway.
    if max_workers == 1 or len(tasks) == 1:
        for t in tasks:
            print(f"\n=== {t[0]} ===")
            _, stats = _worker(t)
            all_stats.extend(stats)
    else:
        with ProcessPoolExecutor(max_workers=max_workers) as pool:
            futures = {pool.submit(_worker, t): t[0] for t in tasks}
            for fut in tqdm(as_completed(futures), total=len(futures), desc="Folders"):
                folder = futures[fut]
                try:
                    _, stats = fut.result()
                    all_stats.extend(stats)
                    print(f"OK   {folder}")
                except Exception as exc:
                    print(f"FAIL {folder} -> {exc}")

    if all_stats:
        stats_path = os.path.join(output_root, "spring_constant_stats_summary.xlsx")
        write_stats_excel(all_stats, stats_path)
    print("\nAll done.")


def main_dual(
    dual_input_folders=None,
    output_root=OUTPUT_ROOT,
    zoom_values=ZOOM_VALUES,
    nm_per_pixel=NM_PER_PIXEL,
    vmin=VMIN, vmax=VMAX,
    min_points=MIN_POINTS,
    mask_to_boundary=MASK_TO_BOUNDARY,
    label1=PROTEIN1_NAME, label2=PROTEIN2_NAME,
    thresh_weak=THRESH_WEAK, thresh_strong=THRESH_STRONG,
    save_images=SAVE_IMAGES,
):
    """
    Dual-label entry point. Reads from CONFIG by default.

    Raises ValueError if VMIN or VMAX is None; auto-derivation is not
    supported in dual-label mode because the colour scale must be set
    identically for both proteins before any per-cell run.
    """
    if dual_input_folders is None:
        dual_input_folders = DUAL_INPUT_FOLDERS
    os.makedirs(output_root, exist_ok=True)
    save_config_snapshot(output_root)

    if vmin is None or vmax is None:
        raise ValueError(
            "Set VMIN and VMAX explicitly in CONFIG for dual-label mode. "
            "Run single-label mode (or the Diagnostic cell) first to find the range."
        )

    print(f"\nColormap scale: log10(k) in [{vmin:.2f}, {vmax:.2f}]")
    print(f"Image generation: {'ON' if save_images else 'OFF (txt tables only)'}")

    for cell_name, (folder1, folder2, rgn_path) in dual_input_folders.items():
        print(f"\n=== {cell_name} ===")
        out = os.path.join(output_root, cell_name)
        os.makedirs(out, exist_ok=True)

        bnd_x_um = bnd_y_um = None
        if rgn_path and os.path.exists(rgn_path):
            bnd_x_px, bnd_y_px = parse_rgn(rgn_path)
            bnd_x_um = (bnd_x_px * nm_per_pixel / 1000).tolist()
            bnd_y_um = (bnd_y_px * nm_per_pixel / 1000).tolist()
            print(f"  Loaded ROI boundary: {len(bnd_x_px)} points")

        run_dual_pipeline(
            cell_name=cell_name,
            folder1=folder1, folder2=folder2,
            output_folder=out,
            zoom_values=zoom_values, nm_per_pixel=nm_per_pixel,
            vmin=vmin, vmax=vmax, min_points=min_points,
            bnd_x_um=bnd_x_um, bnd_y_um=bnd_y_um,
            mask_to_boundary=mask_to_boundary,
            label1=label1, label2=label2,
            thresh_weak=thresh_weak, thresh_strong=thresh_strong,
            save_images=save_images,
        )
    print("\nAll done.")


## 12. Run

Edit the CONFIG cell above, then run this cell.


In [37]:
if N_PROTEINS == 1:
    main()
elif N_PROTEINS == 2:
    main_dual()
else:
    print(f"N_PROTEINS={N_PROTEINS!r} -- set to 1 or 2 in CONFIG")


  -> Config snapshot saved: D:\Publications\Spring-Constant\Spring Constant Home - 6 april 2026\Output\compare_old_vs_new\new\config_snapshot.txt

Auto-deriving colormap scale...
  VMIN <- min log10(k) at zoom=1x across all cells
  VMAX <- max log10(k) at zoom=8x across all cells
  -> VMIN=-3.937, VMAX=4.455
  Paste VMIN=-3.94 and VMAX=4.46 into CONFIG to lock the scale.

Colormap scale: log10(k) in [-3.94, 4.46]
Image generation: ON

=== D:\Publications\Spring-Constant\Input_files_SC\Homer\cell4\loc_and_trc_files ===
  Homer_cell4: 771,194 localisations, 177,353 unique tracks  [frame 201x215 px]
    zoom=1x (160nm/px): 13057 pixels  log10k median=-2.83
      -> saved D:\Publications\Spring-Constant\Spring Constant Home - 6 april 2026\Output\compare_old_vs_new\new\Homer_cell4\Homer_cell4_zoom1x_map.tiff
    zoom=2x (80nm/px): 36888 pixels  log10k median=-1.66
      -> saved D:\Publications\Spring-Constant\Spring Constant Home - 6 april 2026\Output\compare_old_vs_new\new\Homer_cell4\Hom

## 13. Diagnostic Cell (optional)

Quick log10(k) summary at every zoom level for one folder. Useful when
choosing `VMIN`/`VMAX` before a full run.


In [38]:
# Paste a folder path here, or take the first entry of INPUT_FOLDERS by default.
diag_folder = list(INPUT_FOLDERS.keys())[0]

df_diag, _, _ = load_trc_files(diag_folder)
for z in ZOOM_VALUES:
    df_z = compute_spring_constants(df_diag, zoom=z,
                                    nm_per_pixel=NM_PER_PIXEL,
                                    min_points=MIN_POINTS)
    v = df_z["log10k"].replace([float("inf"), float("-inf")], float("nan")).dropna()
    if len(v):
        print(f"zoom={z}x  n={len(v):6d}  min={v.min():.2f}  "
              f"P2={v.quantile(.02):.2f}  median={v.median():.2f}  "
              f"P98={v.quantile(.98):.2f}  max={v.max():.2f}")
    else:
        print(f"zoom={z}x  no valid data")


zoom=1x  n= 10266  min=-3.94  P2=-3.41  median=-2.83  P98=-1.54  max=1.84
zoom=2x  n= 21654  min=-2.77  P2=-2.33  median=-1.66  P98=-0.28  max=2.57
zoom=4x  n= 28860  min=-1.62  P2=-1.25  median=-0.50  P98=1.03  max=3.85
zoom=8x  n= 25266  min=-0.40  P2=-0.10  median=0.69  P98=2.21  max=4.46
